## Cell 1 — Dataset Setup
1. Go to https://universe.roboflow.com/roboflow-universe-projects/license-plate-recognition-rxg4e
2. Click Download → Format: YOLOv8 → Download zip
3. Place the zip in `data/` folder
4. Run the cell below

In [1]:
import zipfile
import os

ZIP_NAME = "License Plate Recognition.v1i.yolov8.zip"
DATA_DIR    = "../data/"
DATASET_DIR = os.path.join(DATA_DIR, "dataset")
zip_path    = os.path.join(DATA_DIR, ZIP_NAME)

if os.path.exists(zip_path):
    os.makedirs(DATASET_DIR, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATASET_DIR)
    print(f"Extracted to: {DATASET_DIR}")
else:
    print(f"Zip not found at: {zip_path}")
    print("Download from Roboflow and place in data/ folder")

print("\nDataset structure:")
for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    if level < 3:
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        for f in files[:3]:
            print(f"{indent}  {f}")

KeyboardInterrupt: 

## Cell 2 — Training
NOTE: CPU training is slow (~2-5 min/epoch). Use Google Colab for GPU speed.

In [ ]:
from ultralytics import YOLO
import os

DATASET_DIR = "../data/dataset"
data_yaml   = None

for root, dirs, files in os.walk(DATASET_DIR):
    for f in files:
        if f.endswith('.yaml'):
            data_yaml = os.path.join(root, f)
            break

if data_yaml is None:
    raise FileNotFoundError("data.yaml not found. Run Cell 1 first.")

print(f"Using: {data_yaml}")

model = YOLO("yolov8n.pt")

results = model.train(
    data=data_yaml,
    epochs=20,
    imgsz=640,
    batch=8,
    patience=10,
    device='cpu',
    project="../runs/train",
    name="alpr_v1",
    exist_ok=True
)

print("Training complete!")
print("Best weights: ../runs/train/alpr_v1/weights/best.pt")

## Cell 3 — Evaluation

In [ ]:
from ultralytics import YOLO
import os

best_pt     = "../runs/train/alpr_v1/weights/best.pt"
DATASET_DIR = "../data/dataset"

if not os.path.exists(best_pt):
    raise FileNotFoundError("Run Cell 2 first.")

model = YOLO(best_pt)

data_yaml = None
for root, dirs, files in os.walk(DATASET_DIR):
    for f in files:
        if f.endswith('.yaml'):
            data_yaml = os.path.join(root, f)
            break

metrics = model.val(data=data_yaml, device='cpu')

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")
print("=" * 50)

## Cell 4 — Inference Test

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2, os, glob

best_pt    = "../runs/train/alpr_v1/weights/best.pt"
sample_dir = "../data/sample_videos/"

if not os.path.exists(best_pt):
    raise FileNotFoundError("Run Cell 2 first.")

model  = YOLO(best_pt)
images = []
for ext in ['*.jpg', '*.jpeg', '*.png']:
    images.extend(glob.glob(os.path.join(sample_dir, ext)))

if not images:
    test_dir = "../data/dataset/test/images/"
    if os.path.exists(test_dir):
        for ext in ['*.jpg', '*.jpeg', '*.png']:
            images.extend(glob.glob(os.path.join(test_dir, ext)))

images = images[:5]

if not images:
    print("No images found. Add images to data/sample_videos/")
else:
    fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 5))
    if len(images) == 1:
        axes = [axes]

    for i, img_path in enumerate(images):
        results   = model(img_path, device='cpu', verbose=False)
        result    = results[0]
        annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)

        axes[i].imshow(annotated)
        axes[i].axis('off')
        axes[i].set_title(os.path.basename(img_path))

        print(f"Image {i+1}: {os.path.basename(img_path)}")
        for box in result.boxes:
            print(f"  Confidence: {float(box.conf[0]):.2f}")
        if not result.boxes:
            print("  No plates detected.")

    plt.tight_layout()
    plt.savefig("../data/sample_inference.png")
    plt.show()

## Cell 5 — Copy Trained Weights to Models Folder

In [ ]:
import shutil, os

src = "../runs/train/alpr_v1/weights/best.pt"
dst = "../models/yolov8_plate.pt"

if not os.path.exists(src):
    raise FileNotFoundError("Run Cell 2 first.")

shutil.copy(src, dst)
size_mb = os.path.getsize(dst) / (1024 * 1024)
print(f"Copied to: {dst}")
print(f"Size: {size_mb:.1f} MB")
print("Restart Streamlit to use trained model!")